# CELL 01

# Knowledge Graph — Notebook 8
## Travel Planning Knowledge Graph — Capstone

### From Knowledge Representation to an Intelligent Travel Planning System

In the previous notebooks, we learned:

    KG-01 → Why Knowledge Graphs?
    KG-02 → Build a KG from Scratch
    KG-03 → Query, Traversal and Multi-Hop Reasoning
    KG-04 → Build a KG from CSV
    KG-05 → KG vs Relational DB vs Vector Store
    KG-06 → Neo4j + Cypher
    KG-07 → RDF + SPARQL

Now we bring these ideas together.

Our problem remains the same:

> "I want to plan a trip using travel knowledge."

This notebook is the capstone.

We will build a small Travel Planning Knowledge Graph
and use it to answer realistic travel questions.

# CELL 02

##  Philosophy

This notebook follows the complete journey:

    REAL-WORLD PROBLEM
           ↓
        KNOWLEDGE
           ↓
       ENTITIES
           ↓
     RELATIONSHIPS
           ↓
        TRIPLES
           ↓
     KNOWLEDGE GRAPH
           ↓
        QUERY
           ↓
      TRAVERSAL
           ↓
       REASONING
           ↓
   TRAVEL RECOMMENDATION

The objective is not merely to write code.

The objective is to understand how a Knowledge Graph
can support reasoning over structured knowledge.

# CELL 03

## The Travel Planning Scenario

Suppose a traveller says:

> "I am planning a weekend trip from Chennai.
> I want a heritage destination nearby.
> I prefer a hotel costing less than ₹3500 per night.
> I also want to know how I can reach the destination."

This is a realistic information need.

Notice that the question contains several requirements:

    Starting location
        ↓
      Chennai

    Destination type
        ↓
      Heritage

    Distance relationship
        ↓
      Near Chennai

    Hotel requirement
        ↓
      Price < ₹3500

    Travel connectivity
        ↓
      How can I reach it?

A Knowledge Graph can connect these pieces of knowledge.

# CELL 04

## Step 1 — Decompose the Question

Before writing code, identify:

### Entities

    Chennai
    Heritage
    Destination
    Hotel

### Relationships

    NEAR
    HAS_CATEGORY
    LOCATED_IN
    CONNECTED_TO
    PRICE_PER_NIGHT

### Conditions

    destination is near Chennai
    destination is Heritage
    hotel price < 3500

This is the first important skill:

> Convert a natural-language information need
> into a graph-oriented information need.

# CELL 05

## Step 2 — Build the Knowledge

We will use the travel knowledge developed
throughout the previous notebooks.

Our knowledge includes:

    Chennai → located in → Tamil Nadu

    Mahabalipuram → near → Chennai

    Mahabalipuram → has category → Heritage

    Pondicherry → near → Chennai

    Pondicherry → has category → Heritage

    Hotel SeaView → located in → Mahabalipuram

    Hotel SeaView → price per night → ₹3500

    Hotel Heritage → located in → Pondicherry

    Hotel Heritage → price per night → ₹3000

    Chennai → connected to → Bengaluru

    Bengaluru → connected to → Mysuru

# CELL 06

## Step 3 — Represent the Knowledge as Triples

The knowledge can be represented as:

    (Chennai, LOCATED_IN, Tamil Nadu)

    (Mahabalipuram, NEAR, Chennai)

    (Mahabalipuram, HAS_CATEGORY, Heritage)

    (Pondicherry, NEAR, Chennai)

    (Pondicherry, HAS_CATEGORY, Heritage)

    (Hotel SeaView, LOCATED_IN, Mahabalipuram)

    (Hotel SeaView, PRICE_PER_NIGHT, 3500)

    (Hotel Heritage, LOCATED_IN, Pondicherry)

    (Hotel Heritage, PRICE_PER_NIGHT, 3000)

    (Chennai, CONNECTED_TO, Bengaluru)

    (Bengaluru, CONNECTED_TO, Mysuru)

In [1]:
# CELL 07

triples = [
    ("Chennai", "LOCATED_IN", "Tamil Nadu"),
    ("Mahabalipuram", "NEAR", "Chennai"),
    ("Mahabalipuram", "HAS_CATEGORY", "Heritage"),
    ("Pondicherry", "NEAR", "Chennai"),
    ("Pondicherry", "HAS_CATEGORY", "Heritage"),
    ("Hotel SeaView", "LOCATED_IN", "Mahabalipuram"),
    ("Hotel SeaView", "PRICE_PER_NIGHT", 3500),
    ("Hotel Heritage", "LOCATED_IN", "Pondicherry"),
    ("Hotel Heritage", "PRICE_PER_NIGHT", 3000),
    ("Chennai", "CONNECTED_TO", "Bengaluru"),
    ("Bengaluru", "CONNECTED_TO", "Mysuru")
]

print("Number of triples:", len(triples))

Number of triples: 11


# CELL 08

## Inspect the Knowledge

Each triple represents one piece of knowledge.

The graph is therefore not magic.

It is constructed from facts such as:

    Entity → Relationship → Entity

or:

    Entity → Relationship → Value

The intelligence comes from being able to connect
these individual facts and answer questions
that may require several relationships.

In [2]:
# CELL 09

for triple in triples:
    print(triple)

('Chennai', 'LOCATED_IN', 'Tamil Nadu')
('Mahabalipuram', 'NEAR', 'Chennai')
('Mahabalipuram', 'HAS_CATEGORY', 'Heritage')
('Pondicherry', 'NEAR', 'Chennai')
('Pondicherry', 'HAS_CATEGORY', 'Heritage')
('Hotel SeaView', 'LOCATED_IN', 'Mahabalipuram')
('Hotel SeaView', 'PRICE_PER_NIGHT', 3500)
('Hotel Heritage', 'LOCATED_IN', 'Pondicherry')
('Hotel Heritage', 'PRICE_PER_NIGHT', 3000)
('Chennai', 'CONNECTED_TO', 'Bengaluru')
('Bengaluru', 'CONNECTED_TO', 'Mysuru')


# CELL 10

## Step 4 — Create the Graph Structure

We will first build a simple graph in Python.

For every subject we will store:

    relationship → object

For example:

    Chennai
       ├── LOCATED_IN → Tamil Nadu
       └── CONNECTED_TO → Bengaluru

In [3]:
# CELL 11

graph = {}

for subject, relationship, object_ in triples:
    if subject not in graph:
        graph[subject] = []
        
    graph[subject].append(
        (relationship, object_)
    )

print("Graph created.")

Graph created.


# CELL 12

## Inspect the Graph

Let us inspect the graph structure.

This is similar to an adjacency-list representation
of a graph.

The important point is:

> A Knowledge Graph can be represented computationally
> as entities connected through meaningful relationships.

In [4]:
# CELL 13

for entity, connections in graph.items():
    print("\n", entity)
    
    for relationship, object_ in connections:
        print("   --", relationship, "-->", object_)


 Chennai
   -- LOCATED_IN --> Tamil Nadu
   -- CONNECTED_TO --> Bengaluru

 Mahabalipuram
   -- NEAR --> Chennai
   -- HAS_CATEGORY --> Heritage

 Pondicherry
   -- NEAR --> Chennai
   -- HAS_CATEGORY --> Heritage

 Hotel SeaView
   -- LOCATED_IN --> Mahabalipuram
   -- PRICE_PER_NIGHT --> 3500

 Hotel Heritage
   -- LOCATED_IN --> Pondicherry
   -- PRICE_PER_NIGHT --> 3000

 Bengaluru
   -- CONNECTED_TO --> Mysuru


## Part 2 — Querying the Knowledge
# CELL 14

## Step 5 — Ask a Simple Question

Question:

> Which destinations are near Chennai?

Translate the question into a graph pattern:

    ?destination
          |
         NEAR
          |
       Chennai

The unknown entity is:

    ?destination

In [5]:
# CELL 15

destinations_near_chennai = []

for subject, relationship, object_ in triples:
    if relationship == "NEAR" and object_ == "Chennai":
        destinations_near_chennai.append(subject)

print("Destinations near Chennai:")

for destination in destinations_near_chennai:
    print("-", destination)

Destinations near Chennai:
- Mahabalipuram
- Pondicherry


# CELL 16

## Observe

We have answered a question by matching
a graph pattern.

This is the fundamental idea behind
graph querying.

    Question
       ↓
    Graph pattern
       ↓
    Matching facts
       ↓
    Answer

##Part 3 — Multi-condition Query
# CELL 17

## Step 6 — Add Another Requirement

Question:

> Which destinations near Chennai are heritage destinations?

Now we need TWO relationships:

    Destination → NEAR → Chennai

AND

    Destination → HAS_CATEGORY → Heritage

This is no longer a single-triple query.

We are combining knowledge from multiple triples.

In [6]:
# CELL 18

near_chennai = set()

heritage_destinations = set()

for subject, relationship, object_ in triples:

    if relationship == "NEAR" and object_ == "Chennai":
        near_chennai.add(subject)

    if relationship == "HAS_CATEGORY" and object_ == "Heritage":
        heritage_destinations.add(subject)

recommended_destinations = (
    near_chennai & heritage_destinations
)

print("Heritage destinations near Chennai:")

for destination in sorted(recommended_destinations):
    print("-", destination)

Heritage destinations near Chennai:
- Mahabalipuram
- Pondicherry


# CELL 19

## Important Observation

We used two separate pieces of knowledge:

    NEAR relationship

and

    HAS_CATEGORY relationship

and combined them.

This is the beginning of graph-based reasoning.

The answer was not stored as a single fact.

Instead, it was derived by combining facts.

## Part 4 — Follow the Graph
# CELL 20

## Step 7 — Find Hotels

Now ask:

> Which hotels are available in these destinations?

We need to follow:

    Destination
          ↓
      LOCATED_IN
          ↑
        Hotel

In our representation:

    Hotel → LOCATED_IN → Destination

In [7]:
# CELL 21

candidate_hotels = []

for subject, relationship, object_ in triples:

    if relationship == "LOCATED_IN":
        if object_ in recommended_destinations:
            candidate_hotels.append(
                (subject, object_)
            )

print("Candidate hotels:")

for hotel, destination in candidate_hotels:
    print(
        "-", hotel,
        "| Destination:", destination
    )

Candidate hotels:
- Hotel SeaView | Destination: Mahabalipuram
- Hotel Heritage | Destination: Pondicherry


## Part 5 — Add Numerical Reasoning

# CELL 22

## Step 8 — Apply the Budget Condition

The traveller said:

> Hotel price must be less than ₹3500.

We therefore need:

    Hotel
       ↓
    PRICE_PER_NIGHT
       ↓
      Price

and:

    Price < 3500

In [8]:
# CELL 23

hotel_prices = {}

for subject, relationship, object_ in triples:

    if relationship == "PRICE_PER_NIGHT":
        hotel_prices[subject] = object_

for hotel, price in hotel_prices.items():
    print(hotel, "→ ₹", price)

Hotel SeaView → ₹ 3500
Hotel Heritage → ₹ 3000


In [9]:
# CELL 24

budget = 3500

affordable_hotels = []

for hotel, destination in candidate_hotels:

    price = hotel_prices.get(hotel)

    if price is not None and price < budget:
        affordable_hotels.append(
            (hotel, destination, price)
        )

print("Affordable hotels:")

for hotel, destination, price in affordable_hotels:
    print(
        hotel,
        "|",
        destination,
        "| ₹", price
    )

Affordable hotels:
Hotel Heritage | Pondicherry | ₹ 3000


# CELL 25

## Important Boundary Condition

Notice the difference between:

    price < 3500

and:

    price <= 3500

The first excludes a hotel costing exactly ₹3500.

The second includes it.

This is a general lesson:

> Knowledge Graphs provide the relationships,
> while query conditions determine how that
> knowledge is used.

## Part 6 — Build a Reusable Query Function
# CELL 26

## Step 9 — Move from One Question to a Reusable Query

Instead of writing new code for every question,
we can create a general triple-matching function.

This is similar to the query function introduced
in KG-03.

In [10]:
# CELL 27

def query_graph(
    triples,
    subject=None,
    relationship=None,
    object_=None
):
    
    results = []

    for s, r, o in triples:

        if subject is not None and s != subject:
            continue

        if relationship is not None and r != relationship:
            continue

        if object_ is not None and o != object_:
            continue

        results.append((s, r, o))

    return results

# CELL 28

## Query Example

Find all facts involving Chennai.

We can specify:

    subject = Chennai

and leave the other conditions open.

In [11]:
# CELL 29

results = query_graph(
    triples,
    subject="Chennai"
)

for result in results:
    print(result)

('Chennai', 'LOCATED_IN', 'Tamil Nadu')
('Chennai', 'CONNECTED_TO', 'Bengaluru')


## Part 7 — Multi-Hop Traversal
# CELL 30

## Step 10 — Multi-Hop Travel Connectivity

Now consider:

    Chennai
       ↓
    CONNECTED_TO
       ↓
    Bengaluru
       ↓
    CONNECTED_TO
       ↓
    Mysuru

Suppose the traveller asks:

> "Can I reach Mysuru from Chennai
> through connected cities?"

This is a multi-hop question.

The answer requires following more than one edge.

# CELL 31

## Why Multi-Hop Questions Matter

A simple database lookup might answer:

    Chennai → CONNECTED_TO → Bengaluru

But the traveller may ask:

> "What is reachable from Chennai
> through multiple connections?"

Now we need graph traversal.

This is where Knowledge Graphs become particularly useful.

In [12]:
# CELL 32

from collections import deque

def bfs_path(graph, start, target):

    queue = deque()
    queue.append((start, [start]))

    visited = set()

    while queue:

        current, path = queue.popleft()

        if current == target:
            return path

        if current in visited:
            continue

        visited.add(current)

        for relationship, neighbor in graph.get(current, []):

            if neighbor not in visited:
                queue.append(
                    (neighbor, path + [neighbor])
                )

    return None

# CELL 33

## Find a Route

We will now search for a path:

    Chennai → Mysuru

In [13]:
# CELL 34

path = bfs_path(
    graph,
    "Chennai",
    "Mysuru"
)

print("Path:", path)

Path: ['Chennai', 'Bengaluru', 'Mysuru']


# CELL 35

## Interpret the Result

If the result is:

    Chennai
       ↓
    Bengaluru
       ↓
    Mysuru

then the Knowledge Graph has given us
a reasoning path.

This is important for explainability.

Instead of simply saying:

    "Mysuru is reachable."

we can show:

    Chennai
       ↓
    Bengaluru
       ↓
    Mysuru

# Part 8 — Add More Travel Knowledge
# CELL 36

## Step 11 — A Knowledge Graph Should Grow

A real Knowledge Graph is rarely complete.

As new information becomes available,
we add new entities and relationships.

Let us extend our travel knowledge.

We will add Hyderabad.

In [14]:
# CELL 37

new_triples = [
    ("Chennai", "CONNECTED_TO", "Hyderabad"),
    ("Hyderabad", "CONNECTED_TO", "Bengaluru"),
    ("Hyderabad", "HAS_CATEGORY", "Heritage"),
    ("Charminar Hotel", "LOCATED_IN", "Hyderabad"),
    ("Charminar Hotel", "PRICE_PER_NIGHT", 2800)
]

triples.extend(new_triples)

print("Total triples:", len(triples))

Total triples: 16


# CELL 38

## Rebuild the Graph

Because we added new knowledge,
we need to update our graph structure.

In [15]:
# CELL 39

graph = {}

for subject, relationship, object_ in triples:

    if subject not in graph:
        graph[subject] = []

    graph[subject].append(
        (relationship, object_)
    )

print("Graph updated.")

Graph updated.


## Part 9 — New Reasoning Question
# CELL 40

## Step 12 — Ask a New Question

Question:

> "Can I travel from Chennai to Mysuru
> through Hyderabad?"

The graph now contains:

    Chennai
       ↓
    Hyderabad
       ↓
    Bengaluru
       ↓
    Mysuru

The graph contains a possible multi-hop path.

The important point is:

> We did not explicitly store
> "Chennai can reach Mysuru through Hyderabad."

It can be derived by traversal.

In [16]:
# CELL 41

path = bfs_path(
    graph,
    "Chennai",
    "Mysuru"
)

print("Path from Chennai to Mysuru:")

if path:
    print(" → ".join(path))
else:
    print("No path found.")

Path from Chennai to Mysuru:
Chennai → Bengaluru → Mysuru


## Part 10 — Complete Recommendation Pipeline
# CELL 42

## Step 13 — Build a Complete Travel Recommendation

Let us now automate the main recommendation logic.

Traveller requirement:

> Find heritage destinations near Chennai
> with hotels costing less than ₹3500.

The reasoning pipeline is:

    1. Find destinations near Chennai
             ↓
    2. Find heritage destinations
             ↓
    3. Find their hotels
             ↓
    4. Find hotel prices
             ↓
    5. Apply budget condition
             ↓
    6. Produce recommendation

In [17]:
# CELL 43

near_chennai = {
    subject
    for subject, relationship, object_ in triples
    if relationship == "NEAR"
    and object_ == "Chennai"
}

heritage_destinations = {
    subject
    for subject, relationship, object_ in triples
    if relationship == "HAS_CATEGORY"
    and object_ == "Heritage"
}

candidate_destinations = (
    near_chennai & heritage_destinations
)

print("Candidate destinations:")

for destination in sorted(candidate_destinations):
    print("-", destination)

Candidate destinations:
- Mahabalipuram
- Pondicherry


In [18]:
# CELL 44

budget = 3500

recommendations = []

for subject, relationship, destination in triples:

    if relationship != "LOCATED_IN":
        continue

    if destination not in candidate_destinations:
        continue

    hotel = subject

    prices = query_graph(
        triples,
        subject=hotel,
        relationship="PRICE_PER_NIGHT"
    )

    if not prices:
        continue

    price = prices[0][2]

    if price < budget:

        recommendations.append(
            {
                "destination": destination,
                "hotel": hotel,
                "price": price
            }
        )

print("Recommendations:")

for recommendation in recommendations:
    print(recommendation)

Recommendations:
{'destination': 'Pondicherry', 'hotel': 'Hotel Heritage', 'price': 3000}


## Part 11 — Explainable Recommendation
# CELL 45

## Step 14 — Do Not Just Give the Answer

An intelligent travel system should ideally
explain WHY a recommendation was made.

For example:

    Destination:
        Pondicherry

    Why selected?
        Pondicherry is near Chennai.
        Pondicherry is a Heritage destination.

    Hotel:
        Hotel Heritage

    Why selected?
        Hotel Heritage is located in Pondicherry.
        Price = ₹3000
        Budget = ₹3500

This is an explainable recommendation.

In [19]:
# CELL 46

for recommendation in recommendations:

    destination = recommendation["destination"]
    hotel = recommendation["hotel"]
    price = recommendation["price"]

    print("\nRecommendation")
    print("----------------------------")

    print("Destination:", destination)

    print(
        "Reason:",
        destination,
        "is near Chennai."
    )

    print(
        "Reason:",
        destination,
        "is a Heritage destination."
    )

    print("Hotel:", hotel)

    print(
        "Reason:",
        hotel,
        "is located in",
        destination
    )

    print("Price: ₹", price)

    print(
        "Budget condition: ₹",
        price,
        "< ₹",
        budget
    )


Recommendation
----------------------------
Destination: Pondicherry
Reason: Pondicherry is near Chennai.
Reason: Pondicherry is a Heritage destination.
Hotel: Hotel Heritage
Reason: Hotel Heritage is located in Pondicherry
Price: ₹ 3000
Budget condition: ₹ 3000 < ₹ 3500


## Part 12 — Question to Graph Pattern
# CELL 47

## Step 15 — From Natural Language to Graph Pattern

Consider:

> "Find an affordable hotel in a heritage
> destination near Chennai."

Do NOT immediately write Python.

First translate the question.

### Natural Language

    affordable hotel

          ↓

    pricePerNight < 3500

### Natural Language

    in a destination

          ↓

    Hotel → LOCATED_IN → Destination

### Natural Language

    heritage destination

          ↓

    Destination → HAS_CATEGORY → Heritage

### Natural Language

    near Chennai

          ↓

    Destination → NEAR → Chennai

The complete graph pattern is:

    Hotel
      |
    LOCATED_IN
      |
    Destination
      |              \
    HAS_CATEGORY     NEAR
      |                \
    Heritage          Chennai

## Part 13 — The Role of LLMs
# CELL 48

## Step 16 — Where Could an LLM Help?

So far, we manually converted:

    Natural Language
          ↓
    Entities
          ↓
    Relationships
          ↓
    Graph Pattern

In a real system, this step could be assisted
by an NLP model or an LLM.

For example:

    User:
    "I want a heritage place near Chennai
     with a budget hotel."

The LLM could help identify:

    Chennai
    Heritage
    Hotel

and relationships such as:

    NEAR
    HAS_CATEGORY
    LOCATED_IN
    PRICE_PER_NIGHT

But an important principle remains:

> The LLM should not replace the Knowledge Graph.

The KG stores structured knowledge.

The LLM can help interpret natural language
and formulate questions.

# CELL 49

## Possible Intelligent KG Architecture

A realistic system could therefore look like:

    User
      |
      ↓
    Natural Language
      |
      ↓
    LLM / NLP
      |
      ↓
    Identify entities and intent
      |
      ↓
    Graph Query
      |
      ↓
    Knowledge Graph
      |
      ↓
    Results
      |
      ↓
    LLM
      |
      ↓
    Natural-language explanation

The Knowledge Graph provides
structured, explicit knowledge.

The LLM provides
language understanding and generation.

## Part 14 — KG + Vector + Relational
# CELL 50

## Step 17 — The Bigger Picture

KG-05 taught us that different representations
are useful for different problems.

### Relational Database

Good for:

    exact values
    filtering
    aggregation
    transactions

### Vector Store

Good for:

    semantic similarity
    meaning-based retrieval
    unstructured text

### Knowledge Graph

Good for:

    explicit relationships
    graph traversal
    multi-hop reasoning
    explainable paths

# CELL 51

## A Real Travel System May Use All Three

Consider:

> "Find me a peaceful heritage destination
> near Chennai with a hotel below ₹3500,
> and explain how I can reach it."

Different parts of the question may require
different representations.

    "peaceful"
        ↓
    Vector Search

    "heritage near Chennai"
        ↓
    Knowledge Graph

    "hotel price < ₹3500"
        ↓
    Relational Database

    "how can I reach it?"
        ↓
    Knowledge Graph

This is why we studied the three representations
in KG-05.

# CELL 52

## Hybrid Architecture

A realistic architecture could therefore be:

                 USER
                   |
                   ↓
                LLM/NLP
                   |
          ┌────────┼────────┐
          ↓        ↓        ↓
       Vector      KG      SQL
       Store              Database
          |        |        |
          └────────┼────────┘
                   ↓
              Combined
               Results
                   ↓
                  LLM
                   ↓
          Travel Recommendation

This is a hybrid Knowledge System.

The important principle is:

> Use the representation that best matches
> each part of the problem.

# CELL 53

## Step 18 — Why Explainability Matters

Suppose the system recommends:

    Pondicherry
    Hotel Heritage

A user may ask:

> "Why did you recommend this?"

The graph can provide the reasoning path:

    Pondicherry
        ↓
      NEAR
        ↓
      Chennai

    Pondicherry
        ↓
    HAS_CATEGORY
        ↓
      Heritage

    Hotel Heritage
        ↓
    LOCATED_IN
        ↓
    Pondicherry

    Hotel Heritage
        ↓
    PRICE_PER_NIGHT
        ↓
       ₹3000

Therefore:

    ₹3000 < ₹3500

The recommendation can be explained
using explicit facts and relationships.

## Part 16 — Knowledge Graph Limitations
# CELL 54

## Step 19 — Knowledge Graphs Are Not Magic

A Knowledge Graph can only reason over
the knowledge that it contains.

Suppose we ask:

> "Is Hotel Heritage open today?"

Our current graph does not contain:

    OPENING_TIME
    CURRENT_STATUS
    AVAILABILITY

Therefore the graph cannot answer this question.

This teaches an important principle:

> The quality of reasoning depends on
> the quality and completeness of knowledge.

# CELL 55

## Other Limitations

A real Travel Planning Knowledge Graph
may need information about:

    weather
    traffic
    train schedules
    flight schedules
    hotel availability
    hotel ratings
    prices
    distances
    opening hours
    holidays
    events

Much of this information changes over time.

Therefore:

> A Knowledge Graph often needs to be
> connected to external data sources.

## Part 17 — Static versus Dynamic Knowledge
# CELL 56

## Static and Dynamic Knowledge

Some travel knowledge does not change (or  slowly change: like Hyderabad is in Andhra Pradesh)

    Chennai is in Tamil Nadu.

Some knowledge changes frequently:

    Hotel price
    Hotel availability
    Traffic
    Weather
    Train availability

Therefore a practical system may combine:

    Knowledge Graph
        +
    Database
        +
    APIs
        +
    Vector Store
        +
    LLM

## Part 18 — Final Architecture
# CELL 57

# Final Travel Planning Architecture

                   USER
                     |
                     ↓
              Natural Language
                     |
                     ↓
                  LLM/NLP
                     |
             ┌───────┼────────┐
             ↓       ↓        ↓
            KG     Vector     SQL
             |       |        |
             |       |        |
             └───────┼────────┘
                     ↓
               Evidence / Facts
                     |
                     ↓
                  Reasoning
                     |
                     ↓
                  LLM
                     |
                     ↓
          Explainable Recommendation

## Part 19 — Technology Mapping
# CELL 58

## What We Have Learned Across the Eight Notebooks

### KG-01

    Why do we need a Knowledge Graph?

### KG-02

    How can we represent knowledge as a graph?

### KG-03

    How can we query and traverse the graph?

### KG-04

    How can we construct a KG from data?

### KG-05

    Which representation is suitable for which question?

### KG-06

    How can we use a real graph database?
    → Neo4j + Cypher

### KG-07

    How can we represent knowledge using
    a standardized semantic model?
    → RDF + SPARQL

### KG-08

    How can we use these ideas to build
    an intelligent Travel Planning system?

## Part 20 — Final Comparison
# CELL 59

## Final Comparison

| Representation | Natural Strength |
|---|---|
| Relational DB | Exact structured data |
| Vector Store | Semantic similarity |
| Knowledge Graph | Explicit relationships |
| RDF | Standardized knowledge representation |
| Neo4j | Graph database and graph traversal |
| Cypher | Querying Neo4j |
| SPARQL | Querying RDF |

There is no universally "best" representation.

The appropriate choice depends on the question.

# CELL 60

#  Challenge

You are now the designer of a Travel Planning
Knowledge Graph system.

A user asks:

> "I am travelling from Chennai.
> I want a heritage destination.
> It should be reasonably close to Chennai.
> I need a hotel below ₹3500 per night.
> I also want to know how I can reach the destination."

Your task:

### Step 1

Identify the entities.

### Step 2

Identify the relationships.

### Step 3

Represent the requirements as graph patterns.

### Step 4

Identify which facts are required.

### Step 5

Query the Knowledge Graph.

### Step 6

Perform multi-hop traversal where required.

### Step 7

Apply the hotel price condition.

### Step 8

Produce an explainable recommendation.

# CELL 61

## Challenge Extension

Now change the question:

> "I want a heritage destination near Chennai,
> but I prefer a peaceful place."

Ask yourself:

Which part of the problem is naturally handled by:

    Knowledge Graph?

Which part might be better handled by:

    Vector Search?

This is a representation-selection problem.

Do not assume that every requirement
must be answered by the Knowledge Graph.

# CELL 62

## Challenge Extension 2

Now consider:

> "Find a heritage destination near Chennai,
> with a hotel below ₹3500, and show the
> shortest route from Chennai."

Identify:

    Knowledge Graph task
    Relational task
    Graph traversal task

Explain why each representation
is appropriate.

## Part 22 — Reflection
# CELL 63

# Reflection

Answer these questions in your own words.

1. What is a Knowledge Graph?

2. What is a triple?

3. What is the difference between an entity
   and a literal?

4. Why are relationships important?

5. What is multi-hop reasoning?

6. How is graph traversal different from
   a simple lookup?

7. Why is explainability easier when
   reasoning paths are explicit?

8. When would you use a relational database?

9. When would you use a vector store?

10. When would you use a Knowledge Graph?

11. What is the difference between
    Neo4j and RDF?

12. What is the difference between
    Cypher and SPARQL?

13. Why might a real AI application
    use KG + Vector Store + Database?

14. What role can an LLM play in a
    Knowledge Graph system?

15. Why can an LLM not simply replace
    the Knowledge Graph?

# CELL 64
 Complete KG Journey

## The Fundamental Idea

    REAL-WORLD KNOWLEDGE
             ↓
        ENTITIES
             ↓
       RELATIONSHIPS
             ↓
          TRIPLES
             ↓
      KNOWLEDGE GRAPH
             ↓
           QUERY
             ↓
        TRAVERSAL
             ↓
         REASONING
             ↓
       EXPLANATION


## Technology Journey

    Python Graph
         ↓
    CSV → Graph
         ↓
    SQLite + Vector Store + KG
         ↓
    Neo4j + Cypher
         ↓
    RDF + SPARQL
         ↓
    Hybrid Intelligent System

# CELL 65

# Final Takeaway

> A Knowledge Graph is not merely a collection
> of nodes and edges.

It is a way of representing knowledge so that
relationships between entities can be explicitly
stored, queried, traversed and reasoned over.

For our Travel Planning problem:

    Knowledge
        ↓
    Relationships
        ↓
    Graph
        ↓
    Query
        ↓
    Multi-hop traversal
        ↓
    Reasoning
        ↓
    Explainable recommendation

And in a modern AI system:

    LLM + Knowledge Graph + Vector Store + Database

can work together.

The central lesson is:

> **Choose the representation according
> to the nature of the problem.**

# CELL 66

# Knowledge Graph Module — Completed

We have moved from:

    "What is a Knowledge Graph?"

to:

    "How can a Knowledge Graph support
     an intelligent application?"

The complete learning progression was:

    CONCEPT
       ↓
    REPRESENTATION
       ↓
    IMPLEMENTATION
       ↓
    QUERY
       ↓
    TRAVERSAL
       ↓
    REASONING
       ↓
    APPLICATION

### Final Question

> If you were asked to design a real
> Travel Planning AI system tomorrow,
> which components would you use,
> and why?

That is the question you should now
be able to answer.